# 04 — Mobility-state analysis

This notebook classifies single-molecule motion into mobility states using frame-to-frame velocities.

Input generated by `03_velocity_analysis.ipynb`:

```text
results/velocity/frame_to_frame_links.csv
results/velocity/velocity_thresholds.csv
```

The notebook performs:
- rule-based slow / fast mobility classification using a control-derived velocity threshold,
- Gaussian mixture model (GMM) classification of log-velocity distributions,
- per-cell mobility-state summaries,
- state-fraction visualizations.

This notebook is an exploratory mobility-state module. It does not replace full HMM inference, which is handled in a separate notebook.


## 1. Imports and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture


plt.style.use("default")


CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

RESULTS_DIR = PROJECT_DIR / "results"
VELOCITY_DIR = RESULTS_DIR / "velocity"
STATE_DIR = RESULTS_DIR / "mobility_states"

FIGURES_DIR = PROJECT_DIR / "figures"
STATE_FIGURES_DIR = FIGURES_DIR / "mobility_states"

for directory in [
    RESULTS_DIR,
    VELOCITY_DIR,
    STATE_DIR,
    FIGURES_DIR,
    STATE_FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


LINKS_PATH = VELOCITY_DIR / "frame_to_frame_links.csv"
THRESHOLDS_PATH = VELOCITY_DIR / "velocity_thresholds.csv"

CONDITION_ORDER = [
    "control",
    "500uM_H2O2",
    "500uM_H2O2_30min",
]

CONDITION_LABELS = {
    "control": "Control",
    "500uM_H2O2": "500 µM H₂O₂",
    "500uM_H2O2_30min": "500 µM H₂O₂ 30 min",
}

N_GMM_STATES = 3
RANDOM_STATE = 42

print("Project directory:", PROJECT_DIR)
print("Links file:", LINKS_PATH)
print("Configuration loaded")


## 2. Load velocity links and thresholds

In [ ]:
if not LINKS_PATH.exists():
    raise FileNotFoundError(
        f"Velocity links file was not found: {LINKS_PATH}. "
        "Run 03_velocity_analysis.ipynb first."
    )

links = pd.read_csv(LINKS_PATH)

required_columns = [
    "protein",
    "condition",
    "sample",
    "cell",
    "file_name",
    "track_id",
    "velocity_um_s",
    "displacement_um",
]

missing_columns = [
    column for column in required_columns
    if column not in links.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}. "
        f"Available columns: {list(links.columns)}"
    )

links = links[
    np.isfinite(links["velocity_um_s"])
    & (links["velocity_um_s"] > 0)
].copy()

links["log10_velocity"] = np.log10(links["velocity_um_s"])

if THRESHOLDS_PATH.exists():
    velocity_thresholds = pd.read_csv(THRESHOLDS_PATH)
else:
    velocity_thresholds = pd.DataFrame()

print("Loaded links:", len(links))
print("Proteins:", sorted(links["protein"].dropna().unique()))

links.head()


## 3. Rule-based slow / fast classification

The simplest mobility-state classifier uses a control-derived velocity threshold.

For each protein, the high-velocity threshold is calculated in `03_velocity_analysis.ipynb` as the selected percentile of the control velocity distribution.

Links above this threshold are classified as `fast`; all other links are classified as `slow_or_typical`.


In [ ]:
def classify_by_control_threshold(links_table, thresholds_table):
    """Classify links into slow_or_typical and fast states using control thresholds."""
    classified = links_table.copy()
    classified["threshold_state"] = "slow_or_typical"
    classified["threshold_um_s"] = np.nan

    for protein in sorted(classified["protein"].dropna().unique()):
        threshold_row = thresholds_table[
            thresholds_table["protein"] == protein
        ]

        if threshold_row.empty:
            print(f"No threshold found for {protein}; skipping threshold classification")
            continue

        threshold = threshold_row["empirical_threshold_um_s"].iloc[0]

        mask = classified["protein"] == protein
        classified.loc[mask, "threshold_um_s"] = threshold

        fast_mask = mask & (classified["velocity_um_s"] > threshold)
        classified.loc[fast_mask, "threshold_state"] = "fast"

    return classified


links_threshold_classified = classify_by_control_threshold(
    links,
    velocity_thresholds,
)

threshold_classified_path = STATE_DIR / "links_threshold_mobility_states.csv"
links_threshold_classified.to_csv(threshold_classified_path, index=False)

print("Saved:", threshold_classified_path)

links_threshold_classified.head()


## 4. Per-cell threshold-state summary

In [ ]:
threshold_summary_rows = []

for keys, group in links_threshold_classified.groupby(
    ["protein", "condition", "sample", "cell", "file_name"],
    observed=True,
):
    protein, condition, sample, cell, file_name = keys

    n_links = len(group)
    n_fast = int((group["threshold_state"] == "fast").sum())
    n_slow = int((group["threshold_state"] == "slow_or_typical").sum())

    threshold_summary_rows.append({
        "protein": protein,
        "condition": condition,
        "sample": sample,
        "cell": cell,
        "file_name": file_name,
        "n_links": n_links,
        "n_fast_links": n_fast,
        "n_slow_or_typical_links": n_slow,
        "fast_fraction": n_fast / n_links if n_links else np.nan,
        "fast_percent": 100 * n_fast / n_links if n_links else np.nan,
        "threshold_um_s": group["threshold_um_s"].dropna().iloc[0]
        if group["threshold_um_s"].notna().any()
        else np.nan,
    })

threshold_cell_summary = pd.DataFrame(threshold_summary_rows)

threshold_cell_summary_path = STATE_DIR / "cell_level_threshold_state_summary.csv"
threshold_cell_summary.to_csv(threshold_cell_summary_path, index=False)

print("Saved:", threshold_cell_summary_path)

threshold_cell_summary.sort_values(
    ["protein", "sample", "cell", "condition"],
    na_position="last",
)


## 5. Gaussian mixture model classification

A Gaussian mixture model (GMM) is fitted to `log10_velocity` separately for each protein.

This provides a data-driven exploratory classification into:
- slow,
- intermediate,
- fast

mobility states.

Important: GMM states are distributional components, not direct mechanistic proof of molecular states.


In [ ]:
def fit_gmm_for_protein(links_table, protein, n_states=3):
    """Fit GMM to log10 velocity for one protein and return classified links."""
    protein_links = links_table[
        links_table["protein"] == protein
    ].copy()

    protein_links = protein_links[
        np.isfinite(protein_links["log10_velocity"])
    ].copy()

    if protein_links.empty:
        return None, pd.DataFrame()

    x = protein_links[["log10_velocity"]].to_numpy()

    model = GaussianMixture(
        n_components=n_states,
        covariance_type="full",
        random_state=RANDOM_STATE,
        n_init=10,
    )

    raw_state = model.fit_predict(x)
    protein_links["gmm_state_raw"] = raw_state

    state_order = (
        protein_links.groupby("gmm_state_raw")["log10_velocity"]
        .mean()
        .sort_values()
        .index
    )

    state_map = {
        old_state: new_state
        for new_state, old_state in enumerate(state_order)
    }

    protein_links["gmm_state"] = protein_links["gmm_state_raw"].map(state_map)

    state_labels = {
        0: "slow",
        1: "intermediate",
        2: "fast",
    }

    protein_links["gmm_state_label"] = protein_links["gmm_state"].map(state_labels)

    return model, protein_links


gmm_models = {}
gmm_results = []

for protein in sorted(links["protein"].dropna().unique()):
    model, result = fit_gmm_for_protein(
        links,
        protein,
        n_states=N_GMM_STATES,
    )

    if model is None:
        continue

    gmm_models[protein] = model
    gmm_results.append(result)

gmm_links = pd.concat(gmm_results, ignore_index=True) if gmm_results else pd.DataFrame()

gmm_links_path = STATE_DIR / "links_gmm_mobility_states.csv"
gmm_links.to_csv(gmm_links_path, index=False)

print("Saved:", gmm_links_path)

gmm_links.head()


## 6. Per-cell GMM-state summary

In [ ]:
gmm_summary_rows = []

if not gmm_links.empty:
    for keys, group in gmm_links.groupby(
        ["protein", "condition", "sample", "cell", "file_name"],
        observed=True,
    ):
        protein, condition, sample, cell, file_name = keys
        n_links = len(group)

        state_counts = group["gmm_state_label"].value_counts()

        gmm_summary_rows.append({
            "protein": protein,
            "condition": condition,
            "sample": sample,
            "cell": cell,
            "file_name": file_name,
            "n_links": n_links,
            "slow_fraction": state_counts.get("slow", 0) / n_links,
            "intermediate_fraction": state_counts.get("intermediate", 0) / n_links,
            "fast_fraction": state_counts.get("fast", 0) / n_links,
            "slow_percent": 100 * state_counts.get("slow", 0) / n_links,
            "intermediate_percent": 100 * state_counts.get("intermediate", 0) / n_links,
            "fast_percent": 100 * state_counts.get("fast", 0) / n_links,
        })

gmm_cell_summary = pd.DataFrame(gmm_summary_rows)

gmm_cell_summary_path = STATE_DIR / "cell_level_gmm_state_summary.csv"
gmm_cell_summary.to_csv(gmm_cell_summary_path, index=False)

print("Saved:", gmm_cell_summary_path)

gmm_cell_summary.sort_values(
    ["protein", "sample", "cell", "condition"],
    na_position="last",
)


## 7. Plotting functions

In [ ]:
def setup_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2)


def get_available_conditions(data):
    return [
        condition for condition in CONDITION_ORDER
        if condition in data["condition"].unique()
    ]


def plot_threshold_fast_fraction(protein):
    data = threshold_cell_summary[
        threshold_cell_summary["protein"] == protein
    ].copy()

    if data.empty:
        print(f"No threshold-state data for {protein}")
        return

    data["condition"] = pd.Categorical(
        data["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    data = data.sort_values(["sample", "cell", "condition"])
    x_map = {condition: i for i, condition in enumerate(CONDITION_ORDER)}

    fig, ax = plt.subplots(figsize=(7, 5))

    for (sample, cell), group in data.groupby(["sample", "cell"], observed=True):
        group = group.sort_values("condition")
        x = group["condition"].map(x_map).astype(float).to_numpy()
        y = group["fast_percent"].to_numpy()

        ax.plot(
            x,
            y,
            marker="o",
            linewidth=1.5,
            alpha=0.55,
            label=f"S{sample} cell{cell}",
        )

    median_summary = (
        data.groupby("condition", observed=True)["fast_percent"]
        .median()
        .reindex(CONDITION_ORDER)
    )

    ax.plot(
        range(len(CONDITION_ORDER)),
        median_summary.values,
        marker="o",
        linewidth=4,
        color="black",
        label="group median",
    )

    ax.set_xticks(range(len(CONDITION_ORDER)))
    ax.set_xticklabels(
        [CONDITION_LABELS[c] for c in CONDITION_ORDER],
        rotation=20,
    )

    ax.set_ylabel("Fast links per cell, %")
    ax.set_title(f"{protein}: threshold-based high-mobility fraction")

    setup_axis(ax)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
    plt.tight_layout()

    output_path = STATE_FIGURES_DIR / f"{protein}_threshold_fast_fraction.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_gmm_state_fractions(protein):
    data = gmm_links[
        gmm_links["protein"] == protein
    ].copy()

    if data.empty:
        print(f"No GMM-state data for {protein}")
        return

    counts = (
        data.groupby(["condition", "gmm_state_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(CONDITION_ORDER)
        .fillna(0)
    )

    fractions = counts.div(counts.sum(axis=1), axis=0)

    state_order = ["slow", "intermediate", "fast"]

    fig, ax = plt.subplots(figsize=(7, 5))
    bottom = np.zeros(len(fractions))

    for state in state_order:
        if state not in fractions.columns:
            continue

        ax.bar(
            range(len(fractions)),
            fractions[state].values,
            bottom=bottom,
            label=state,
        )
        bottom += fractions[state].values

    ax.set_xticks(range(len(CONDITION_ORDER)))
    ax.set_xticklabels(
        [CONDITION_LABELS[c] for c in CONDITION_ORDER],
        rotation=20,
    )

    ax.set_ylabel("Fraction of links")
    ax.set_title(f"{protein}: GMM mobility-state fractions")
    ax.legend(frameon=False)

    setup_axis(ax)
    plt.tight_layout()

    output_path = STATE_FIGURES_DIR / f"{protein}_gmm_state_fractions.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


def plot_gmm_log_velocity_components(protein):
    data = gmm_links[
        gmm_links["protein"] == protein
    ].copy()

    if data.empty or protein not in gmm_models:
        print(f"No GMM model for {protein}")
        return

    model = gmm_models[protein]

    x_min = data["log10_velocity"].quantile(0.001)
    x_max = data["log10_velocity"].quantile(0.999)
    x_grid = np.linspace(x_min, x_max, 600).reshape(-1, 1)

    log_prob = model.score_samples(x_grid)
    total_density = np.exp(log_prob)

    responsibilities = model.predict_proba(x_grid)
    component_densities = responsibilities * total_density[:, np.newaxis]

    fig, ax = plt.subplots(figsize=(7, 5))

    ax.hist(
        data["log10_velocity"],
        bins=80,
        density=True,
        alpha=0.4,
        label="Observed log10 velocity",
    )

    ax.plot(
        x_grid[:, 0],
        total_density,
        linewidth=3,
        color="black",
        label="GMM total",
    )

    for state_index in range(component_densities.shape[1]):
        ax.plot(
            x_grid[:, 0],
            component_densities[:, state_index],
            linewidth=2,
            label=f"component {state_index}",
        )

    ax.set_xlabel("log10 velocity, velocity in µm/s")
    ax.set_ylabel("Probability density")
    ax.set_title(f"{protein}: GMM components")
    ax.legend(frameon=False)

    setup_axis(ax)
    plt.tight_layout()

    output_path = STATE_FIGURES_DIR / f"{protein}_gmm_components.png"
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", output_path)


## 8. Generate mobility-state figures

In [ ]:
proteins = sorted(links["protein"].dropna().unique())

for protein in proteins:
    plot_threshold_fast_fraction(protein)
    plot_gmm_state_fractions(protein)
    plot_gmm_log_velocity_components(protein)


## 9. Quick interpretation guide

- Threshold-based classification is simple and reproducible. It uses a control-derived high-velocity threshold for each protein.
- GMM-based classification is exploratory and data-driven. It can reveal slow, intermediate, and fast components in the velocity distribution.
- State fractions should be interpreted carefully: components are statistical mobility classes, not direct proof of molecular mechanisms.
- Cell-level summaries are preferred for condition-level comparison because trajectories inside the same cell are not independent.


## 10. Output files

This notebook generates:

```text
results/mobility_states/links_threshold_mobility_states.csv
results/mobility_states/cell_level_threshold_state_summary.csv
results/mobility_states/links_gmm_mobility_states.csv
results/mobility_states/cell_level_gmm_state_summary.csv
figures/mobility_states/*_threshold_fast_fraction.png
figures/mobility_states/*_gmm_state_fractions.png
figures/mobility_states/*_gmm_components.png
```

These outputs can be used in final figure generation and may serve as a bridge toward full hidden Markov model analysis.
